# M3L3 E24 - Multiagente HR + IT con orquestador híbrido (estilo PIM3)

Este ejercicio es una versión simplificada de la arquitectura del **Proyecto Integrador M3 (PIM3)**: un orquestador que combina reglas por palabras clave con un LLM de respaldo, y agentes especialistas que responden con RAG básico.

Diferencia con E23 (`multiagente_rag_simple`): acá el orquestador no depende solo de keywords. Si las keywords no alcanzan para decidir, un LLM clasifica con salida estructurada, igual que hace el PIM3 real.

Objetivo de la clase:
- Entender por qué un orquestador híbrido (reglas + LLM) es más robusto que solo reglas o solo LLM.
- Ver cómo se modela `trace_steps` para que el sistema sea auditable, paso previo a la observabilidad de M3L4.
- Practicar el mismo patrón que usa el PIM3, pero con 2 dominios (HR, IT) en vez de 3, y sin ChromaDB.

Idea central:

```text
Usuario pregunta
      |
      v
orquestador: keywords primero -> LLM estructurado si es ambiguo
      |
      v
agente especialista (hr o tech) recupera contexto simple (RAG)
      |
      v
LLM responde usando solo ese contexto
```

Este ejercicio es deliberadamente simple: recuperación por palabras compartidas en vez de embeddings, y sin ChromaDB. El objetivo es que el patrón de orquestación quede clarísimo antes de sumar esas capas.

## Paso 1 - Instalar dependencias y crear el modelo

Usamos:

- `langgraph`: para armar el flujo como grafo.
- `langchain-openai`: para llamar a OpenAI desde LangChain.
- `pydantic`: para la salida estructurada del clasificador (`RouteDecision`), igual que en el PIM3.

Este notebook funciona **con o sin API key**. Si no cargás una, el orquestador queda limitado a las reglas por keywords (sin el respaldo del LLM) y los agentes devuelven el contexto recuperado sin generar respuesta — así podés probar el routing y el grafo sin gastar tokens, igual que el modo offline del PIM3.

In [1]:
# Instalamos solo lo necesario para este ejercicio.
!pip install langgraph langchain-openai pydantic -q

import os
import re
import unicodedata
from getpass import getpass
from typing import Literal, TypedDict

from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END

# La API key es opcional: si la dejás vacía, el notebook corre en modo offline.
api_key = os.environ.get("OPENAI_API_KEY") or getpass("OpenAI API Key (vacío = modo offline): ").strip()
HAS_API_KEY = bool(api_key)

if HAS_API_KEY:
    os.environ["OPENAI_API_KEY"] = api_key
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
else:
    llm = None

print("Modo:", "LLM real (routing híbrido completo)" if HAS_API_KEY else "offline (solo keywords + RAG simple, sin LLM)")

Modo: LLM real (routing híbrido completo)


## Paso 2 - Definir el estado compartido

Igual que en el PIM3, sumamos `trace_steps`: cada nodo agrega un paso legible que explica qué hizo. Esto es lo que en M3L4 vas a conectar con Langfuse para observabilidad real; acá lo vemos en su forma más simple, como lista de diccionarios.

Campos:
- `query`: pregunta original del usuario.
- `intent`: dominio elegido por el orquestador (`hr`, `tech` o `unknown`).
- `reason`: motivo breve de esa decisión.
- `context`: fragmentos recuperados por el agente especialista.
- `answer`: respuesta final.
- `trace_steps`: bitácora de pasos, uno por nodo ejecutado.

In [2]:
class AgentState(TypedDict):
    query: str
    intent: str
    reason: str
    context: str
    answer: str
    trace_steps: list[dict]

print("State definido")

State definido


## Paso 3 - Base de conocimiento simple (HR + IT)

Dos dominios, como pediste: `hr` (Recursos Humanos) y `tech` (Soporte IT).

El retriever es el mismo RAG simple de E23: cuenta palabras compartidas entre la consulta y cada documento. El PIM3 real hace esto mismo pero contra ChromaDB con embeddings; acá alcanza para explicar la idea sin esa infraestructura.

In [3]:
DOCS = {
    "hr": [
        "Vacaciones: cada empleado tiene 15 días hábiles por año, a coordinar con el manager.",
        "Licencias: las licencias por estudio o enfermedad se piden en PeopleOps con al menos 48hs de aviso.",
        "Beneficios: el seguro médico y el bono de desempeño se activan a partir del día 1.",
        "Onboarding: el plan de las primeras dos semanas lo arma RRHH junto al manager directo.",
    ],
    "tech": [
        "VPN: reiniciar el cliente, validar el 2FA y abrir un ticket si el problema persiste.",
        "Contraseña: se restablece desde el portal de identidad, sin pasar por soporte.",
        "Notebook: reportar equipo dañado o perdido con el número de serie al equipo de IT.",
        "Accesos: los permisos de sistemas se piden por ticket, aprobados por el manager del área.",
    ],
}


def tokenize(text: str) -> set[str]:
    return set(re.findall(r"[a-záéíóúñ0-9]{3,}", text.lower()))


def retrieve_docs(domain: str, query: str, k: int = 2) -> list[str]:
    query_tokens = tokenize(query)
    scored = []
    for doc in DOCS.get(domain, []):
        score = len(query_tokens & tokenize(doc))
        scored.append((score, doc))
    scored.sort(key=lambda item: item[0], reverse=True)
    return [doc for score, doc in scored[:k]]

print("Base de conocimiento y retriever simple listos")

Base de conocimiento y retriever simple listos


## Paso 4 - Orquestador híbrido (el corazón del ejercicio)

Así rutea el PIM3 real (`route_query` en `src/agents.py`), simplificado a 2 dominios:

1. **Keywords primero.** Si la consulta matchea señales de un solo dominio, se resuelve ahí mismo — rápido, gratis y determinístico.
2. **Si mezcla dominios**, se manda a `unknown` en vez de forzar una respuesta desde el agente equivocado.
3. **Si no hay ninguna señal clara**, recién ahí se usa el LLM, con salida estructurada (`RouteDecision`, un modelo Pydantic) para que la clasificación sea siempre un valor válido y no texto libre a interpretar.
4. **Sin API key**, el paso 3 no está disponible: una consulta sin keywords cae directo en `unknown` (modo offline).

Esto es más robusto que E23 (solo keywords) y más barato que clasificar todo con LLM.

In [4]:
Intent = Literal["hr", "tech", "unknown"]

KEYWORDS: dict[str, list[str]] = {
    "hr": ["vacacion", "vacaciones", "licencia", "beneficio", "bono", "desempeno", "rrhh", "recursos humanos", "onboarding"],
    "tech": ["vpn", "2fa", "doble factor", "contrasena", "password", "notebook", "soporte", "acceso", "wifi", "login"],
}


def normalize_text(text: str) -> str:
    # Saca tildes para que "contraseña" y "contrasena" matcheen igual.
    normalized = unicodedata.normalize("NFKD", text.lower())
    return "".join(char for char in normalized if not unicodedata.combining(char))


def keyword_matches(query: str) -> list[str]:
    q = normalize_text(query)
    return [domain for domain, words in KEYWORDS.items() if any(word in q for word in words)]


class RouteDecision(BaseModel):
    intent: Intent = Field(description="Dominio elegido: hr, tech o unknown.")
    reason: str = Field(description="Motivo breve del ruteo.")


def route_query(query: str) -> RouteDecision:
    matched = keyword_matches(query)

    if len(matched) == 1:
        domain = matched[0]
        return RouteDecision(intent=domain, reason=f"Se detectaron señales claras de {domain} por palabras clave.")

    if len(matched) > 1:
        return RouteDecision(intent="unknown", reason="La consulta mezcla HR y IT; no se puede resolver con un solo agente.")

    if not HAS_API_KEY:
        return RouteDecision(intent="unknown", reason="Sin señales de keywords y sin LLM disponible para desambiguar (modo offline).")

    prompt = ChatPromptTemplate.from_messages([
        (
            "system",
            "Clasificá la consulta interna en hr, tech o unknown. "
            "hr incluye vacaciones, licencias, beneficios, bonos, desempeño y onboarding. "
            "tech incluye VPN, contraseñas, 2FA, notebooks, soporte y accesos. "
            "Si no está claro o no es ninguno de los dos, usa unknown. Devolvé salida estructurada.",
        ),
        ("human", "{query}"),
    ])
    structured_llm = llm.with_structured_output(RouteDecision)
    return (prompt | structured_llm).invoke({"query": query})


def orchestrator_node(state: AgentState) -> dict:
    decision = route_query(state["query"])
    trace_steps = [
        {"step": "Entrada", "detail": "Recibí la consulta del usuario."},
        {"step": "Clasificación", "detail": f"El orquestador eligió '{decision.intent}'. Motivo: {decision.reason}"},
    ]
    return {"intent": decision.intent, "reason": decision.reason, "trace_steps": trace_steps}

print("Orquestador híbrido definido")

Orquestador híbrido definido


## Paso 5 - Agentes especialistas (HR y Tech)

Cada agente:

1. Recupera documentos de su dominio con `retrieve_docs`.
2. Si hay LLM disponible, genera la respuesta usando solo ese contexto.
3. Si no hay LLM (modo offline), devuelve el contexto recuperado sin redactar respuesta — igual que el PIM3 cuando no hay `OPENAI_API_KEY`.
4. Registra cada paso en `trace_steps`, para que el flujo completo quede auditable.

In [5]:
def answer_with_context(domain: str, state: AgentState) -> dict:
    trace_steps = list(state.get("trace_steps", []))
    trace_steps.append({"step": "Agente seleccionado", "detail": f"La consulta pasa al agente especialista '{domain}'."})

    docs = retrieve_docs(domain, state["query"], k=2)
    context = "\n".join(f"- {doc}" for doc in docs)
    trace_steps.append({"step": "Búsqueda RAG", "detail": f"Recuperé {len(docs)} fragmentos relevantes de la base de {domain}."})

    if not HAS_API_KEY:
        trace_steps.append({"step": "Modo offline", "detail": "No hay OPENAI_API_KEY; se omite la generación con LLM."})
        return {
            "context": context,
            "answer": f"[modo offline] Contexto recuperado, pero no hay LLM para redactar la respuesta:\n{context}",
            "trace_steps": trace_steps,
        }

    prompt = (
        f"Sos un agente especialista de {domain}.\n"
        "Respondé en español usando únicamente el contexto provisto.\n"
        "Si falta un dato puntual, decilo y respondé igual con la política aplicable.\n\n"
        f"Contexto:\n{context}\n\n"
        f"Consulta: {state['query']}\n\n"
        "Respuesta breve y accionable:"
    )
    response = llm.invoke(prompt)
    trace_steps.append({"step": "Generación", "detail": "El LLM redactó la respuesta usando solo el contexto recuperado."})

    return {"context": context, "answer": response.content.strip(), "trace_steps": trace_steps}


def hr_node(state: AgentState) -> dict:
    return answer_with_context("hr", state)


def tech_node(state: AgentState) -> dict:
    return answer_with_context("tech", state)


def unknown_node(state: AgentState) -> dict:
    trace_steps = list(state.get("trace_steps", []))
    trace_steps.append({"step": "Fallback", "detail": "No hay un dominio claro (HR o IT); no se consulta RAG."})
    return {
        "context": "",
        "answer": "No tengo suficiente información para derivar esta consulta. Puedo ayudarte con temas de RR.HH. o soporte técnico.",
        "trace_steps": trace_steps,
    }

print("Agentes definidos")

Agentes definidos


## Paso 6 - Armar el grafo

Mismo esqueleto que `graph.py` del PIM3: `orchestrator` decide, `add_conditional_edges` rutea, cada agente termina en `END`.

```text
START
  |
  v
orchestrator_node
  |--------- hr_node
  |--------- tech_node
  |--------- unknown_node
              |
              v
             END
```

In [6]:
def next_node(state: AgentState) -> str:
    intent = state.get("intent", "unknown")
    if intent in {"hr", "tech"}:
        return intent
    return "unknown"


graph = StateGraph(AgentState)

graph.add_node("orchestrator", orchestrator_node)
graph.add_node("hr", hr_node)
graph.add_node("tech", tech_node)
graph.add_node("unknown", unknown_node)

graph.add_edge(START, "orchestrator")
graph.add_conditional_edges(
    "orchestrator",
    next_node,
    {"hr": "hr", "tech": "tech", "unknown": "unknown"},
)
for node in ["hr", "tech", "unknown"]:
    graph.add_edge(node, END)

app = graph.compile()
print("Grafo compilado")

Grafo compilado


## Paso 7 - Ejecutar consultas de prueba

Seis casos que muestran los cuatro caminos del orquestador:

- Keyword clara de HR.
- Keyword clara de Tech.
- Consulta mezclada (HR + Tech) -> `unknown` por ambigüedad, sin pasar por LLM.
- Consulta sin keywords -> depende del LLM para desambiguar (o cae en `unknown` si estás en modo offline).
- Consulta totalmente fuera de alcance -> `unknown`.

Para cada una imprimimos `trace_steps` completo: es la misma idea de bitácora que en M3L4 vas a mandar a Langfuse.

In [7]:
def make_initial_state(query: str) -> AgentState:
    return {"query": query, "intent": "", "reason": "", "context": "", "answer": "", "trace_steps": []}


def run_query(query: str):
    result = app.invoke(make_initial_state(query))
    print("=" * 80)
    print("Consulta:", query)
    print("Intent:  ", result["intent"])
    print("Trace:")
    for step in result["trace_steps"]:
        print(f"  - [{step['step']}] {step['detail']}")
    print("Respuesta:", result["answer"])
    return result


queries = [
    "¿Cuántos días de vacaciones tengo por año?",
    "No puedo conectarme a la VPN de la empresa",
    "Necesito pedir vacaciones pero también se me rompió la notebook",
    "¿Quién revisa las solicitudes de home office?",
    "¿Cuál es la capital de Francia?",
]

for q in queries:
    run_query(q)

Consulta: ¿Cuántos días de vacaciones tengo por año?
Intent:   hr
Trace:
  - [Entrada] Recibí la consulta del usuario.
  - [Clasificación] El orquestador eligió 'hr'. Motivo: Se detectaron señales claras de hr por palabras clave.
  - [Agente seleccionado] La consulta pasa al agente especialista 'hr'.
  - [Búsqueda RAG] Recuperé 2 fragmentos relevantes de la base de hr.
  - [Generación] El LLM redactó la respuesta usando solo el contexto recuperado.
Respuesta: Tienes 15 días hábiles de vacaciones por año, los cuales debes coordinar con tu manager.
Consulta: No puedo conectarme a la VPN de la empresa
Intent:   tech
Trace:
  - [Entrada] Recibí la consulta del usuario.
  - [Clasificación] El orquestador eligió 'tech'. Motivo: Se detectaron señales claras de tech por palabras clave.
  - [Agente seleccionado] La consulta pasa al agente especialista 'tech'.
  - [Búsqueda RAG] Recuperé 2 fragmentos relevantes de la base de tech.
  - [Generación] El LLM redactó la respuesta usando solo el conte

## Checks automáticos

Estos checks usan solo casos de keywords (no dependen de tener API key), para que corran igual en modo online y offline.

In [8]:
def run_checks():
    r1 = app.invoke(make_initial_state("quiero pedir vacaciones"))
    assert r1["intent"] == "hr", r1

    r2 = app.invoke(make_initial_state("no anda mi vpn"))
    assert r2["intent"] == "tech", r2

    r3 = app.invoke(make_initial_state("vacaciones y también falla la vpn"))
    assert r3["intent"] == "unknown", r3

    print("Checks E24 OK")


run_checks()

Checks E24 OK


## Cierre - Qué construiste y cómo se compara con el PIM3

| Concepto | Este ejercicio | PIM3 real |
|---|---|---|
| Dominios | hr, tech | hr, tech, finance |
| Routing | keywords -> LLM estructurado si es ambiguo | igual, misma lógica (`route_query`) |
| Retrieval | conteo de palabras compartidas | embeddings + ChromaDB |
| Observabilidad | `trace_steps` en memoria | `trace_steps` + Langfuse |
| Sin API key | modo offline (keywords + contexto, sin generación) | modo offline equivalente |

Frase clave:

> El orquestador híbrido no reemplaza al LLM ni depende ciegamente de él: usa reglas cuando alcanzan y el LLM solo cuando hace falta. Eso es lo que hace al PIM3 rápido, barato y predecible en los casos comunes.

Próximas mejoras posibles (fuera de este ejercicio):
- Sumar `finance` como tercer dominio.
- Reemplazar `retrieve_docs` por embeddings + vector store (ver M3L2 E07-E09).
- Conectar `trace_steps` a Langfuse (ver M3L4).
- Ver la implementación completa en `PIM3/src/agents.py` y `PIM3/src/graph.py`.